# Notebook 02: Implement and Validate Reward Functions

**Agency Calculus Empirical Validation — Paper C**

This notebook:
1. Loads and tests all three reward functions (SUM, NASH, JAM)
2. Demonstrates the structural differences between them
3. Shows how to hook them into the AI Economist planner
4. Verifies JAM's singularity is preserved (no epsilon)


In [ ]:
# ── Environment check ─────────────────────────────────────────────────────
# If ai_economist is missing, run notebook 01 first (it handles installation
# and the required kernel restart).
import sys, os

try:
    import ai_economist  # noqa: F401
except ModuleNotFoundError:
    raise SystemExit(
        "\n❌  ai_economist not found. Run notebook 01_setup_and_test first,\n"
        "    restart the kernel, then return here."
    )

# Add src/ to path
for candidate in [
    '/content/ac-validation/src',
    os.path.join(os.getcwd(), '..', 'src'),
    os.path.join(os.getcwd(), 'src'),
]:
    if os.path.exists(candidate) and candidate not in sys.path:
        sys.path.insert(0, candidate)
        print(f'src on path: {candidate}')
        break


In [ ]:
import sys, os, math
import numpy as np
import matplotlib.pyplot as plt

sys.path.insert(0, os.path.join(os.getcwd(), '..', 'src'))

from ac_rewards import (
    sum_reward, nash_reward, jam_reward,
    jam_reward_epsilon, jam_reward_softmin, softmin,
    get_reward_fn, REWARD_FUNCTIONS,
)
print('Reward functions loaded:', list(REWARD_FUNCTIONS.keys()))

## 1. Basic Correctness Tests

In [ ]:
# Test vectors
equal = [5.0, 5.0, 5.0, 5.0]          # Equal distribution
skewed = [20.0, 10.0, 5.0, 1.0]       # SUM prefers sacrificing the floor
near_zero = [100.0, 100.0, 100.0, 0.001]  # Near-floor case

for label, utils in [('equal', equal), ('skewed', skewed), ('near_zero', near_zero)]:
    print(f'\nUtilities: {utils}  (floor={min(utils):.4f})')
    print(f'  SUM  = {sum_reward(utils):>12.4f}')
    print(f'  NASH = {nash_reward(utils):>12.4f}')
    print(f'  JAM  = {jam_reward(utils):>12.4f}')

In [ ]:
# JAM zero-utility guard
zero_case = [10.0, 10.0, 10.0, 0.0]
print(f'JAM with zero utility: {jam_reward(zero_case)}')
print(f'Expected: -1e10 = {-1e10}')
assert jam_reward(zero_case) == -1e10, 'JAM zero guard failed!'
print('Zero guard: OK')

## 2. Compensation Mechanism Demonstration

This is the core theoretical point of Paper A: SUM permits compensation (gains for high-utility agents can offset losses for low-utility agents), NASH permits bounded compensation, JAM permits none.

In [ ]:
# Compensation test: start from [5, 5, 5, 5]
# Transfer delta from agent 3 (floor) to agents 0,1,2
# Does the planner reward increase? It should for SUM, not for JAM.

base = [5.0, 5.0, 5.0, 5.0]
deltas = np.linspace(0, 4.9, 100)  # how much we take from agent 3

sum_rewards, nash_rewards, jam_rewards = [], [], []

for d in deltas:
    # Take d from agent 3, give d/3 to each of agents 0,1,2
    u = [5 + d/3, 5 + d/3, 5 + d/3, 5 - d]
    sum_rewards.append(sum_reward(u))
    nash_rewards.append(nash_reward(u))
    jam_rewards.append(jam_reward(u))

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, vals, title, color in zip(
    axes,
    [sum_rewards, nash_rewards, jam_rewards],
    ['SUM: unbounded compensation', 'NASH: bounded compensation', 'JAM: zero compensation'],
    ['#e74c3c', '#f39c12', '#2ecc71'],
):
    ax.plot(deltas, vals, color=color, linewidth=2)
    ax.axhline(vals[0], color='gray', linestyle='--', alpha=0.5, label='baseline')
    ax.set_xlabel('Transfer from floor agent')
    ax.set_ylabel('Planner reward')
    ax.set_title(title)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

plt.suptitle('Compensation Mechanism: How Each Objective Responds to Floor Compression', y=1.02)
plt.tight_layout()
plt.savefig('../results/compensation_mechanism.png', bbox_inches='tight', dpi=150)
plt.show()

print(f'SUM increases as floor drops: {sum_rewards[-1] > sum_rewards[0]} (expected True)')
print(f'NASH decreases as floor drops: {nash_rewards[-1] < nash_rewards[0]} (expected True)')
print(f'JAM  decreases as floor drops: {jam_rewards[-1] < jam_rewards[0]} (expected True)')
print(f'JAM decreases MORE than NASH: {(nash_rewards[0] - nash_rewards[-1]) < (jam_rewards[0] - jam_rewards[-1])} (expected True)')

## 3. JAM Gradient Analysis

Key theoretical property: the gradient of log(x) as x→0+ is +∞.
This is the infinite cost that prevents the planner from driving any agent to zero.

In [ ]:
floor_values = np.linspace(0.001, 5.0, 1000)

# Reward values
jam_vals = np.log(floor_values)
jam_eps_vals = np.log(floor_values + 0.1)

# Gradients (d/d_floor)
jam_grad = 1.0 / floor_values
jam_eps_grad = 1.0 / (floor_values + 0.1)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(floor_values, jam_vals, color='#2ecc71', label='JAM: log(x)', linewidth=2)
ax1.plot(floor_values, jam_eps_vals, color='#27ae60', linestyle='--',
         label='JAM+ε: log(x+0.1)', linewidth=2)
ax1.axvline(0.1, color='gray', linestyle=':', alpha=0.5, label='x=0.1 (epsilon)')
ax1.set_xlabel('Floor agent utility  min(u_i)')
ax1.set_ylabel('Planner reward')
ax1.set_title('JAM Reward vs Floor Utility')
ax1.legend()
ax1.spines['top'].set_visible(False)
ax1.spines['right'].set_visible(False)

ax2.plot(floor_values, jam_grad, color='#2ecc71', label='JAM gradient: 1/x', linewidth=2)
ax2.plot(floor_values, jam_eps_grad, color='#27ae60', linestyle='--',
         label='JAM+ε gradient: 1/(x+0.1)', linewidth=2)
ax2.set_ylim(0, 50)
ax2.axvline(0.1, color='gray', linestyle=':', alpha=0.5)
ax2.set_xlabel('Floor agent utility  min(u_i)')
ax2.set_ylabel('|∂R/∂floor|')
ax2.set_title('Gradient Magnitude Near Zero\n(JAM → ∞; JAM+ε → 1/ε = 10)')
ax2.legend()
ax2.spines['top'].set_visible(False)
ax2.spines['right'].set_visible(False)

plt.tight_layout()
plt.savefig('../results/jam_gradient_analysis.png', bbox_inches='tight', dpi=150)
plt.show()

# Key numbers
print(f'JAM gradient at x=0.01: {1/0.01:.0f}  (→ ∞ as x → 0)')
print(f'JAM+ε gradient at x=0.01: {1/(0.01+0.1):.2f}  (finite, ≈ 1/ε)')
print(f'\nConclusion: epsilon destroys the infinite cost — finite cost = finite compensation window')

## 4. AI Economist Planner Hook

The AI Economist planner receives per-episode coin rewards. We override the planner's reward computation to inject our objective function.

In [ ]:
def get_planner_reward(env_state: dict, condition: str) -> float:
    """
    Extract worker utilities from env state and compute planner reward.
    
    In the AI Economist, agent utility = coin earned this episode.
    We use the cumulative coin holdings as a proxy for utility.
    
    Args:
        env_state: Info dict returned by env.step()
        condition: 'sum', 'nash', or 'jam'
    """
    reward_fn = get_reward_fn(condition)
    
    # Extract per-agent coin earned (utility proxy)
    # The AI Economist returns per-agent rewards in the rewards dict
    # We use coin holdings from the env state
    utilities = []
    for agent_idx in range(env_state.get('n_agents', 4)):
        agent_key = str(agent_idx)
        coin = env_state.get('agents', {}).get(agent_key, {}).get('coin', 1.0)
        utilities.append(max(coin, 1e-8))  # ensure positive for log
    
    return reward_fn(utilities)


class ACPlannerRewardWrapper:
    """
    Wraps the AI Economist environment to override the planner reward.
    
    Usage in RLlib training:
        env = RLlibEnvWrapper(env_config)
        wrapper = ACPlannerRewardWrapper(env, condition='jam')
        
        # In the training loop, after env.step():
        rewards['p'] = wrapper.compute_planner_reward(env.env)
    """
    
    def __init__(self, env, condition: str):
        self.env = env
        self.condition = condition
        self.reward_fn = get_reward_fn(condition)
    
    def compute_planner_reward(self, agent_rewards: dict) -> float:
        """
        Compute planner reward from per-agent utilities.
        
        Args:
            agent_rewards: dict mapping agent_id -> reward from this step
        """
        # Collect worker (non-planner) utilities
        utilities = [
            max(float(r), 1e-8)
            for k, r in agent_rewards.items()
            if k != 'p'  # exclude planner
        ]
        if not utilities:
            return 0.0
        return self.reward_fn(utilities)


# Smoke test
dummy_rewards = {'0': 10.0, '1': 5.0, '2': 8.0, '3': 2.0, 'p': 0.0}
for condition in ['sum', 'nash', 'jam']:
    wrapper = ACPlannerRewardWrapper(None, condition)
    r = wrapper.compute_planner_reward(dummy_rewards)
    print(f'{condition.upper()} planner reward: {r:.4f}')

## 5. Summary

- SUM: reward increases when floor agent is taxed to benefit majority ✓
- NASH: reward decreases as floor drops (bounded compensation) ✓  
- JAM: gradient → ∞ as floor → 0 (zero compensation, infinite cost) ✓
- JAM+ε: gradient capped at 1/ε — structural guarantee destroyed ✓

**Next:** Notebook 03 — implement and validate POLI agency proxy computation.